In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

# 读取 .env 配置文件中的信息，相关的环境变量以 .env文件中的优先
load_dotenv(override=True)

custom_profile = {
    "max_input_tokens": 128_000
}
AGNES_AI_API_KEY = os.getenv("AGNES_AI_API_KEY")
AGNES_AI_BASE_URL = os.getenv("AGNES_AI_BASE_URL")
AGNES_AI_MODEL = os.getenv("AGNES_AI_MODEL")

model = init_chat_model(
    model_provider='openai',
    model=AGNES_AI_MODEL,
    api_key=AGNES_AI_API_KEY,
    base_url=AGNES_AI_BASE_URL,
    profile=custom_profile
)

messages = [
    SystemMessage(content="你是个非常友好的AI助手"),
    HumanMessage(content="你好啊，我是老王，你是谁？"),
    AIMessage(content="你好老王，我是小王"),
    HumanMessage(content="好的小王，很高兴认识你"),
    AIMessage(content="你高兴得太早了"),
    HumanMessage(content="呵呵，你什么意思")
]

agent = create_agent(
    model=model,
    middleware= [
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001)
            ]
        )
    ]
)

response = agent.invoke({
    "messages": messages
})
for msg in response["messages"]:
    msg.pretty_print()

In [12]:
agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001)
            ],
            keep=("messages",2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)
response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

历史消息摘要：用户老王与助手小王进行了初次问候和自我介绍，双方表达了友好的态度，对话氛围轻松愉快。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈，开个玩笑嘛！我是说如果刚见面就太热情、把气氛搞得太严肃，反而会显得不太自然。

既然咱们之前聊得很轻松，那我也就不端着啦。老王，刚才你说“高兴得太早”，是觉得我哪句话太贫嘴了，还是有什么具体的事要跟我念叨念叨？😄
